# step 1 - Config

In [0]:
# ============================================================
# Config
# ============================================================

spark.conf.set('spark.databricks.photon.enabled', 'false')
spark.conf.set('spark.sql.parquet.enableVectorizedReader', 'false')

storage_account = 'your-storage-account'
client_id       = 'your-client-id'
tenant_id       = 'your-tenant-id'
client_secret   = 'your-client-secret'

spark.conf.set(f'fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net', 'OAuth')
spark.conf.set(f'fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net', 'org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider')
spark.conf.set(f'fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net', client_id)
spark.conf.set(f'fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net', client_secret)
spark.conf.set(f'fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net', f'https://login.microsoftonline.com/{tenant_id}/oauth2/token')

gold = 'abfss://gold@adlslogisticsstore.dfs.core.windows.net/'

from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import *
from datetime import datetime

print('Config complete!')

Config complete!


#  step 1 Create Audit Log Delta Table

In [0]:
# ============================================================
# Create Audit Log Delta Table
# ============================================================

# Define audit log schema
audit_schema = StructType([
    StructField('pipeline_name',     StringType(),    False),
    StructField('notebook_name',     StringType(),    False),
    StructField('layer',             StringType(),    False),
    StructField('status',            StringType(),    False),
    StructField('records_read',      LongType(),      True),
    StructField('records_written',   LongType(),      True),
    StructField('records_rejected',  LongType(),      True),
    StructField('error_message',     StringType(),    True),
    StructField('start_time',        TimestampType(), True),
    StructField('end_time',          TimestampType(), True),
    StructField('duration_seconds',  LongType(),      True),
    StructField('run_date',          StringType(),    True)
])

# Create empty audit table
df_audit_empty = spark.createDataFrame([], audit_schema)

df_audit_empty.write \
    .format('delta') \
    .mode('overwrite') \
    .option('path', f'{gold}audit_log/') \
    .saveAsTable('logistics_db.audit_log')

print('Audit log table created!')
spark.sql('DESCRIBE logistics_db.audit_log').show(truncate=False)

Audit log table created!
+----------------+---------+-------+
|col_name        |data_type|comment|
+----------------+---------+-------+
|pipeline_name   |string   |NULL   |
|notebook_name   |string   |NULL   |
|layer           |string   |NULL   |
|status          |string   |NULL   |
|records_read    |bigint   |NULL   |
|records_written |bigint   |NULL   |
|records_rejected|bigint   |NULL   |
|error_message   |string   |NULL   |
|start_time      |timestamp|NULL   |
|end_time        |timestamp|NULL   |
|duration_seconds|bigint   |NULL   |
|run_date        |string   |NULL   |
+----------------+---------+-------+



In [0]:
# ============================================================
# Audit Log Helper Function
# Call this function at start and end of every pipeline
# ============================================================

from datetime import datetime

def write_audit_log(
    pipeline_name,
    notebook_name,
    layer,
    status,
    records_read      = 0,
    records_written   = 0,
    records_rejected  = 0,
    error_message     = None,
    start_time        = None,
    end_time          = None
):
    # Calculate duration
    if start_time and end_time:
        duration = int((end_time - start_time).total_seconds())
    else:
        duration = 0

    # Create audit record
    audit_data = [(
        pipeline_name,
        notebook_name,
        layer,
        status,
        records_read,
        records_written,
        records_rejected,
        error_message,
        start_time,
        end_time,
        duration,
        datetime.now().strftime('%Y-%m-%d')
    )]

    audit_schema = StructType([
        StructField('pipeline_name',    StringType(),    False),
        StructField('notebook_name',    StringType(),    False),
        StructField('layer',            StringType(),    False),
        StructField('status',           StringType(),    False),
        StructField('records_read',     LongType(),      True),
        StructField('records_written',  LongType(),      True),
        StructField('records_rejected', LongType(),      True),
        StructField('error_message',    StringType(),    True),
        StructField('start_time',       TimestampType(), True),
        StructField('end_time',         TimestampType(), True),
        StructField('duration_seconds', LongType(),      True),
        StructField('run_date',         StringType(),    True)
    ])

    df_audit = spark.createDataFrame(audit_data, audit_schema)

    df_audit.write \
        .format('delta') \
        .mode('append') \
        .option('path', f'{gold}audit_log/') \
        .saveAsTable('logistics_db.audit_log')

    print(f'Audit log written — {pipeline_name} | {status}')

print('write_audit_log function ready!')

write_audit_log function ready!


In [0]:
# Test Audit Log with Real Pipeline Run

In [0]:
# ============================================================
# Test Audit Log with Real Pipeline Run
# ============================================================

from datetime import datetime

# Simulate a pipeline run with audit logging
pipeline_name = 'pl_ingest_delivery_records'
notebook_name = 'nb_silver_cleanse_deliveries'
layer         = 'silver'

# Record start time
start_time = datetime.now()
print(f'Pipeline started: {start_time}')

try:
    # Simulate reading records
    records_read     = 787060
    records_written  = 745796
    records_rejected = records_read - records_written

    # Record end time
    end_time = datetime.now()

    # Write SUCCESS audit log
    write_audit_log(
        pipeline_name    = pipeline_name,
        notebook_name    = notebook_name,
        layer            = layer,
        status           = 'SUCCESS',
        records_read     = records_read,
        records_written  = records_written,
        records_rejected = records_rejected,
        error_message    = None,
        start_time       = start_time,
        end_time         = end_time
    )

except Exception as e:
    end_time = datetime.now()

    # Write FAILED audit log
    write_audit_log(
        pipeline_name    = pipeline_name,
        notebook_name    = notebook_name,
        layer            = layer,
        status           = 'FAILED',
        records_read     = 0,
        records_written  = 0,
        records_rejected = 0,
        error_message    = str(e),
        start_time       = start_time,
        end_time         = end_time
    )
    raise

# View audit log
print('\n=== Audit Log ===')
spark.sql('''
    SELECT pipeline_name, notebook_name, layer,
           status, records_read, records_written,
           records_rejected, duration_seconds, run_date
    FROM logistics_db.audit_log
    ORDER BY start_time DESC
''').show(truncate=False)

Pipeline started: 2026-02-23 19:53:04.167412
Audit log written — pl_ingest_delivery_records | SUCCESS

=== Audit Log ===
+--------------------------+----------------------------+------+-------+------------+---------------+----------------+----------------+----------+
|pipeline_name             |notebook_name               |layer |status |records_read|records_written|records_rejected|duration_seconds|run_date  |
+--------------------------+----------------------------+------+-------+------------+---------------+----------------+----------------+----------+
|pl_ingest_delivery_records|nb_silver_cleanse_deliveries|silver|SUCCESS|787060      |745796         |41264           |0               |2026-02-23|
+--------------------------+----------------------------+------+-------+------------+---------------+----------------+----------------+----------+



# **Add Gold Layer Audit Entry**

In [0]:
# ============================================================
# Add Gold Layer Audit Entry
# ============================================================

from datetime import datetime

start_time = datetime.now()

try:
    write_audit_log(
        pipeline_name    = 'pl_ingest_delivery_records',
        notebook_name    = 'nb_gold_delta_tables',
        layer            = 'gold',
        status           = 'SUCCESS',
        records_read     = 745796,
        records_written  = 745796,
        records_rejected = 0,
        error_message    = None,
        start_time       = start_time,
        end_time         = datetime.now()
    )

except Exception as e:
    write_audit_log(
        pipeline_name    = 'pl_ingest_delivery_records',
        notebook_name    = 'nb_gold_delta_tables',
        layer            = 'gold',
        status           = 'FAILED',
        records_read     = 0,
        records_written  = 0,
        records_rejected = 0,
        error_message    = str(e),
        start_time       = start_time,
        end_time         = datetime.now()
    )
    raise

# View complete audit log
print('=== Complete Audit Log ===')
spark.sql('''
    SELECT pipeline_name, notebook_name, layer,
           status, records_read, records_written,
           records_rejected, duration_seconds, run_date
    FROM logistics_db.audit_log
    ORDER BY start_time ASC
''').show(truncate=False)

Audit log written — pl_ingest_delivery_records | SUCCESS
=== Complete Audit Log ===
+--------------------------+----------------------------+------+-------+------------+---------------+----------------+----------------+----------+
|pipeline_name             |notebook_name               |layer |status |records_read|records_written|records_rejected|duration_seconds|run_date  |
+--------------------------+----------------------------+------+-------+------------+---------------+----------------+----------------+----------+
|pl_ingest_delivery_records|nb_silver_cleanse_deliveries|silver|SUCCESS|787060      |745796         |41264           |0               |2026-02-23|
|pl_ingest_delivery_records|nb_gold_delta_tables        |gold  |SUCCESS|745796      |745796         |0               |0               |2026-02-23|
+--------------------------+----------------------------+------+-------+------------+---------------+----------------+----------------+----------+

